# MediCheck-GPT

This notebook runs my whole project end to end on a Colab T4. Data and checkpoints are saved to my Drive as they are produced, and every stage that has already finished is skipped, so I can pick up wherever a session stopped.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # confirm the T4 is attached

In [ ]:
from google.colab import drive
drive.mount('/content/drive')                     # the raw CMS file lives in my Drive
ROOT = '/content/drive/MyDrive/build-transformer'   # same folder as 01_data.ipynb

In [ ]:
%cd /content
!rm -rf /content/medicheck-gpt                       # start from a clean copy of the code
!git clone https://github.com/cece15/medicheck-gpt.git /content/medicheck-gpt
%cd /content/medicheck-gpt
!pip install -q -r requirements.txt
# data and checkpoints live in my Drive, so nothing is lost if the session disconnects
!mkdir -p "$ROOT/medicheck/data" "$ROOT/medicheck/checkpoints"
!ln -sfn "$ROOT/medicheck/data" data && ln -sfn "$ROOT/medicheck/checkpoints" checkpoints && ls -l
!git log --oneline -1                               # the version of the code this run used

## 1. Data

I turn each row of the CMS file into one English sentence and split the data by hospital, not by row: 160 hospitals for training, 20 for validation and 20 for testing. The test hospitals never appear in training, so the scores show whether the model learned what a record looks like rather than memorizing one hospital's lines.

In [ ]:
import os
if not os.path.exists('data/processed/test.txt'):
    !python gpt.py --input "$ROOT/data/raw/MUP_INP_RY26_P03_V10_DY24_PrvSvc.CSV" --prepare data/processed
!head -c 600 data/processed/train.txt

## 2. Baseline: bigram

The bigram predicts the next character from the current one only. If the transformer can't beat this, it hasn't learned anything.

In [ ]:
if not os.path.exists('checkpoints/bigram.pth'):
    !python gpt.py --input data/processed --train checkpoints/bigram.pth --model bigram --context_size 8 --lr 1e-2 --steps 3000

## 3. Transformer

4 layers, 4 heads, 128-dimensional embeddings and a 271-character window, which is the longest record plus its two newlines, so the model always sees one whole record. My first prototype overfit after about 1,500 steps, so here I use a cosine learning-rate schedule with warmup, weight decay and early stopping.

In [ ]:
if not os.path.exists('checkpoints/gpt.pth'):
    !python gpt.py --input data/processed --train checkpoints/gpt.pth --steps 10000 --eval_every 500 --patience 4

In [ ]:
# loss curve
import json, matplotlib.pyplot as plt
h = json.load(open('checkpoints/gpt_history.json'))
plt.plot([r['step'] for r in h], [r['train'] for r in h], label='train')
plt.plot([r['step'] for r in h], [r['val'] for r in h], label='validation (unseen hospitals)')
plt.xlabel('step'); plt.ylabel('cross-entropy loss'); plt.legend(); plt.title('MediCheck-GPT training')
plt.savefig('checkpoints/loss_curve.png', dpi=200, bbox_inches='tight'); plt.show()

## 3a. Model size

Same data, same seed, same early stopping; only the size and dropout change. I trained the base model three times in total, which is how I could tell real differences from run-to-run noise.

In [ ]:
# smaller and larger than the base model (base = 4 layers, 4 heads, 128-dim, already trained above)
if not os.path.exists('checkpoints/gpt_large.pth'):
    !python gpt.py --input data/processed --train checkpoints/gpt_small.pth --n_layer 2 --n_head 2 --n_embd 64 --steps 6000
    !python gpt.py --input data/processed --train checkpoints/gpt_large.pth --n_layer 6 --n_head 6 --n_embd 192 --dropout 0.2 --steps 6000

In [ ]:
# which size does best on hospitals it has never seen?
import math, torch, pandas as pd
from src.model import build, count_params
rows = []
for name in ['gpt_small', 'gpt', 'gpt_large']:
    ck = torch.load(f'checkpoints/{name}.pth', map_location='cpu')
    best = min(ck['history'], key=lambda r: r['val'])            # the checkpoint holds the best evaluation
    rows.append({'model': name, **{k: ck['config'][k] for k in ['n_layer', 'n_head', 'n_embd', 'dropout']},
                 'parameters': count_params(build(ck['kind'], **ck['config'])),
                 'best_step': best['step'], 'best_val_loss': round(best['val'], 4),
                 'val_perplexity': round(math.exp(best['val']), 3), 'seconds': best['seconds']})
pd.DataFrame(rows)

## 3b. LLM teacher

Kacper suggested having an LLM write reference sentences and training my small model to reproduce them. I use Qwen2.5-1.5B-Instruct on the T4. In my first attempt the LLM invented numbers in 15% of sentences, and many of the sentences that passed attached real numbers to the wrong label. So now Python computes every number, the LLM gets each fact as a full phrase with one example sentence, and any sentence with an invented or mislabeled number is rejected. A rejected record gets one retry; if it fails again, I leave it out.

In [ ]:
import os
if not os.path.exists('data/described_v2/test.txt'):       # the teacher is the slow step, so it only runs once
    !python gpt.py --input data/processed --teacher data/described_v2 --limit 5000
print(open('data/described_v2/teacher_report.json').read())
!head -c 1200 data/described_v2/train.txt

## 3c. Student, version 2: pre-train, then fine-tune

My first student trained from scratch on random windows and learned the teacher's style while ignoring the record. This version starts from my record model, which already knows what records look like, and fine-tunes on whole examples with the loss counted on the explanation only. Validation loss here covers the explanation only, so it can't be compared with the first student's.

In [ ]:
if not os.path.exists('checkpoints/explain_v2.pth'):
    !python gpt.py --input data/described_v2 --train checkpoints/explain_v2.pth --init_from checkpoints/gpt.pth --steps 4000 --eval_every 250 --patience 4

In [ ]:
# the record goes in as a Python argument because the shell would treat its '$' signs as variables
import subprocess
rec = open('data/processed/test.txt').read().split('\n')[1]
print('record :', rec)
print(subprocess.run(['python', 'gpt.py', '--eval', 'checkpoints/explain_v2.pth', '--prompt', rec, '--n', '3', '--temperature', '0.5'],
                     capture_output=True, text=True).stdout)

## 3d. Student, version 3: placeholders

Version 2 wrote like the teacher but filled in numbers from memory: none of its sentences used the record's numbers. A model this small doesn't copy digits from its input. So I replace the record's own values in the teacher's sentences with placeholders (`{BILL}`, `{TOTAL}`, `{HOSPITAL}` and so on). The student learns where each placeholder goes, and code puts the real values back. This is delexicalisation, the usual way data-to-text systems keep numbers faithful.

In [ ]:
if not os.path.exists('checkpoints/explain_v3.pth'):
    !python gpt.py --input data/described_v2 --train checkpoints/explain_v3.pth --init_from checkpoints/gpt.pth --delex --steps 4000 --eval_every 250 --patience 4
print(subprocess.run(['python', 'gpt.py', '--eval', 'checkpoints/explain_v3.pth', '--prompt', rec, '--n', '3', '--temperature', '0.5'],
                     capture_output=True, text=True).stdout)

## 3e. Student, version 4: the condition and the state as placeholders too

Testing on Illinois showed version 3 still writing two facts from memory. The state was copied as a fixed word ("Waukegan, MI"), and the condition was often wrong, because the teacher rephrased conditions and only exact matches became placeholders. Version 4 also turns the teacher's condition phrase and the state into placeholders, so code fills them from the record, and the checks now test both.

In [ ]:
if not os.path.exists('checkpoints/explain_v4.pth'):
    !python gpt.py --input data/described_v2 --train checkpoints/explain_v4.pth --init_from checkpoints/gpt.pth --delex --steps 4000 --eval_every 250 --patience 4
print(subprocess.run(['python', 'gpt.py', '--eval', 'checkpoints/explain_v4.pth', '--prompt', rec, '--n', '3', '--temperature', '0.5'],
                     capture_output=True, text=True).stdout)

## 4. Generating records

In [ ]:
!python gpt.py --eval checkpoints/gpt.pth --n 3
!python gpt.py --eval checkpoints/gpt.pth --n 2 --prompt "Trinity Health Livonia Hospital in Livonia, MI billed DRG "

## 5. Evaluation

Perplexity, next-character accuracy, ROUGE-L, BLEU and BERTScore (rescaled), plus my own checks: whether generated records follow the rules real records follow, and whether the explainer's sentences pass the same checks as the teacher's.

In [ ]:
# the record models (results are saved per dataset, e.g. gpt_processed_eval.json)
if not os.path.exists('checkpoints/gpt_processed_eval.json'):
    !python gpt.py --input data/processed --evaluate checkpoints/bigram.pth
    !python gpt.py --input data/processed --evaluate checkpoints/gpt.pth --hf_metrics
# the explainers, versions 3 and 4, on the same test sentences and the same checks
!python gpt.py --input data/described_v2 --evaluate checkpoints/explain_v3.pth --hf_metrics
!python gpt.py --input data/described_v2 --evaluate checkpoints/explain_v4.pth --hf_metrics

## 6. Flagging silent errors

I plant four kinds of error in records from the test hospitals and ask two questions. Ranking (ROC AUC): across different records, do the corrupted ones score higher? Noticing (paired): on the same record, does the error raise the model's surprise? 0.5 is chance; below 0.5 means the model is getting it backwards.

In [ ]:
if not os.path.exists('checkpoints/gpt_processed_flag.json'):
    !python gpt.py --input data/processed --flag checkpoints/gpt.pth --background checkpoints/bigram.pth
    !python gpt.py --input data/processed --flag checkpoints/gpt_large.pth --background checkpoints/bigram.pth
print(open('checkpoints/gpt_processed_flag.json').read())

## 6b. Demo file

Five real test records and five with planted errors, shuffled. Rules catch the arithmetic errors first, and the model ranks the rest by how surprising it finds them.

In [ ]:
import random, sys
sys.path.insert(0, '.')
from src.flag import corrupt, KINDS
from src.data import parse
test = [r for r in open('data/processed/test.txt').read().split('\n') if parse(r)]
train = [r for r in open('data/processed/train.txt').read().split('\n') if parse(r)]
drg = {parse(r)['drg']: parse(r)['desc'] for r in train}
cities = {}
for r in train:
    f = parse(r); cities.setdefault(f['state'], set()).add(f['city'])
cities = {s: sorted(c) for s, c in cities.items()}
rng = random.Random(7)
clean = rng.sample(test, 5)
planted = [corrupt(r, KINDS[i % 4], drg, cities, rng) for i, r in enumerate(rng.sample(test, 5))]
demo = clean + [p for p in planted if p]
rng.shuffle(demo)
open('data/suspicious.txt', 'w').write('\n'.join(demo) + '\n')
open('data/suspicious_answer_key.txt', 'w').write('\n'.join(('PLANTED  ' if r in planted else 'clean    ') + r for r in demo))
!python gpt.py --input data/processed --flag checkpoints/gpt.pth --flag_file data/suspicious.txt   
!cat data/suspicious_answer_key.txt

## 8. Testing

Three levels. The unit and end-to-end tests check the logic and that every stage runs. The new-state test checks whether the models learned what a Medicare record is or just memorized Michigan and Ohio: Illinois is a state none of them has seen.

In [ ]:
# unit tests and the end-to-end test (CPU, about a minute)
!python -m pip install -q pytest
!python -m pytest tests -q

In [ ]:
# Illinois: never seen in training
if not os.path.exists('data/state_il/test.txt'):
    !python gpt.py --input "$ROOT/data/raw/MUP_INP_RY26_P03_V10_DY24_PrvSvc.CSV" --prepare data/state_il --states IL
# 1. does the record model still predict records well?  (Michigan and Ohio test perplexity: 1.94-1.96)
!python gpt.py --input data/state_il --evaluate checkpoints/gpt.pth
# 2. does the explainer still write correct sentences?  (compare with Michigan and Ohio)
!python gpt.py --eval checkpoints/explain_v4.pth --explain_file data/state_il/test.txt --n 100 --temperature 0.5 | tail -12
# 3. does the flagger still notice planted errors?
!python gpt.py --input data/state_il --flag checkpoints/gpt.pth

## 7. Files in Drive (`build-transformer/medicheck/`)

In [ ]:
!ls -lh checkpoints/ data/described_v2/

## Demo commands

`!python gpt.py --eval checkpoints/explain_v4.pth --interactive --temperature 0.5` turns a pasted record into a plain-English sentence.

`!python gpt.py --input data/processed --flag checkpoints/gpt.pth --flag_file data/suspicious.txt` checks a file with the rules first, then ranks the rest by the model.